# 🧬 Promoter Variant Transformer Demo
### *Non-Coding Oncogenic Mutation Prediction & In Silico Saturation Mutagenesis*

Inspired by **DeepMind's Enformer (Avsec et al., Nature 2021)** and **Nucleotide Transformer (Dalla-Torre et al., Nature Biotech 2023)**, this notebook demonstrates:
1. Encoding 512 bp core human promoter sequences with 1D Conv Stem + **Rotary Position Embeddings (RoPE)**.
2. Quantifying transcriptional reactivation from somatic non-coding driver mutations: **TERT `C228T`** and **`C250T`**.
3. Running **In Silico Saturation Mutagenesis (ISM)** to map the functional regulatory landscape at single-nucleotide resolution.

In [ ]:
import os
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import TERT_PROMOTER_WT, TERT_MUTATIONS, MODEL_CONFIG, TERT_MUTATION_POS_C228T, TERT_MUTATION_POS_C250T
from src.dna_tokenizer import DNATokenizer
from src.promoter_transformer import PromoterTransformerModel
from src.saturation_mutagenesis import InSilicoSaturationMutagenesis
from src.visualizer import PromoterVisualizer

print("PyTorch Version:", torch.__version__)
print("TERT Core Promoter Length:", len(TERT_PROMOTER_WT), "bp")

## 1. Load Pre-Trained RoPE Promoter Transformer

In [ ]:
tokenizer = DNATokenizer(max_length=MODEL_CONFIG['max_seq_len'])
model = PromoterTransformerModel(MODEL_CONFIG)

ckpt_path = '../checkpoints/best_promoter_model.pt'
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    print("Loaded trained weights from:", ckpt_path)
else:
    print("Using randomly initialized model (run train_and_evaluate.py to train!)")
model.eval()

## 2. Score Clinical TERT Driver Mutations (`C228T` & `C250T`)

In [ ]:
ism = InSilicoSaturationMutagenesis(model=model, tokenizer=tokenizer)

# Score C228T (Hotspot 1 - Glioblastoma, Bladder, Melanoma)
score_c228t = ism.score_single_variant(TERT_PROMOTER_WT, position=TERT_MUTATION_POS_C228T, alt_nucleotide='T')
# Score C250T (Hotspot 2 - Melanoma)
score_c250t = ism.score_single_variant(TERT_PROMOTER_WT, position=TERT_MUTATION_POS_C250T, alt_nucleotide='T')
# Score benign control C228A
score_c228a = ism.score_single_variant(TERT_PROMOTER_WT, position=TERT_MUTATION_POS_C228T, alt_nucleotide='A')

print("TERT Wild-Type Baseline Activity:  ", f"{score_c228t['wt_activity']:.3f}")
print("TERT C228T Mutated Activity:       ", f"{score_c228t['mut_activity']:.3f}", f"({score_c228t['fold_change']:.2f}x induction)")
print("TERT C250T Mutated Activity:       ", f"{score_c250t['mut_activity']:.3f}", f"({score_c250t['fold_change']:.2f}x induction)")
print("TERT C228A Benign Control Activity:", f"{score_c228a['mut_activity']:.3f}", f"({score_c228a['fold_change']:.2f}x induction)")

## 3. Visualize In Silico Saturation Mutagenesis Heatmap

In [ ]:
scan = ism.run_saturation_scan(TERT_PROMOTER_WT, start_pos=215, end_pos=260, batch_size=32)
viz = PromoterVisualizer(output_dir='../figures')
heatmap_path = viz.plot_saturation_mutagenesis_heatmap(
    ism_matrix=scan['ism_matrix'],
    positions=scan['positions'],
    ref_bases=scan['ref_bases'],
    filename='demo_tert_saturation_mutagenesis.png'
)

from IPython.display import Image
Image(heatmap_path)